# LR Universe: OOF Factory for Ensemble Diversity — Churn Prediction (PS S6E3)
------

## Introduction

This notebook builds a logistic regression (LR) framework to serve as an OOF (out-of-fold) prediction factory for a future ensemble. The answer to why invest time in LR is diversity. The value of a model in an ensemble is not determined by its score, but how much it reduces the ensemble's mistakes. In order to reduce these mistakes, your model must be reasonable good, and it must make different mistakes from the other models. 

LR operates in differently to tree-based models. Trees partion the feature space into rectangles. LR draws a single hyperplane in the transformed feature space. So LR's errors are structural different from tree errors. 

This notebook generates multiple diverse OOF predictions by using:
- Diverse feature preprocessing like splines, polynomials, interactions, binning, tree-derived features.
- Different categorical encodings like OHE, target encoding, WoE, James-Stain, CatBoost, LOO, binary, and Helmert
- Different types of regularization like L2 Ridge, L1 Lasso, and ElasticNet

Before each experiment, we display the transformation applied in form of diagram so you can see exactly what we do.

## Setup

The notebook has been configure to have a `TEST_MODE` for testing. This allows you to test, debug, and iterate through ideas without waiting too much. Keep into account that AUC scores in `TEST_MODE`are meaningless for comparison given they are computed on a tiny sample. 

We also fixed a random seed for everything to ensure reproducibility and to aligned the same fold structure to avoid leakage using a metalearner. 

In [1]:
import warnings
warnings.filterwarnings("ignore")

TEST_MODE = True          
TEST_SAMPLE_N = 200      
TEST_FOLDS = 3             

DSEED = 10431053
FOLDS_FULL = 5
SAVE_DIR_NAME = "./save"

In [2]:
import pandas as pd
import numpy as np
import os, time
from pathlib import Path

from sklearn.preprocessing import (
    StandardScaler, SplineTransformer, OneHotEncoder,
    OrdinalEncoder, PolynomialFeatures, TargetEncoder,
    FunctionTransformer,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier

This notebook gives you the option to run cuML that provides logistic regression on the GPU, which is much faster than CPU. 

In [3]:
# RAPIDS GPU
try:
    import cuml
    from cuml.linear_model import LogisticRegression as cuLogisticRegression
    from cuml.preprocessing import StandardScaler as cuStandardScaler
    HAS_CUML = True
except ImportError:
    HAS_CUML = False

import torch
USE_GPU = HAS_CUML and torch.cuda.is_available()
print(f"TEST_MODE: {TEST_MODE}  |  RAPIDS/cuML: {HAS_CUML}  |  USE_GPU: {USE_GPU}")

# Category Encoders
try:
    import category_encoders as ce
    HAS_CE = True
except ImportError:
    HAS_CE = False

print(f"category_encoders available: {HAS_CE}")
if not HAS_CE:
    print("  Install with: pip install category_encoders")

# feature_engine
!pip install feature_engine -q
try:
    from feature_engine.encoding import CountFrequencyEncoder
    from feature_engine.discretisation import (
        EqualFrequencyDiscretiser,
        DecisionTreeDiscretiser,
    )
    from feature_engine.creation import DecisionTreeFeatures
    HAS_FE = True
except ImportError:
    HAS_FE = False

print(f"feature_engine available: {HAS_FE}")


TEST_MODE: True  |  RAPIDS/cuML: True  |  USE_GPU: True


category_encoders available: True


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/243.5 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 4.7 MB/s eta 0:00:00


feature_engine available: True


## Load Data

Three datasets:

- **train** (~594k rows): competition training set with target `Churn`.
- **test** (~255k rows): competition test set, no target — predictions submitted here.
- **original** (~7k rows): the Telco Customer Churn dataset (possibly also synthetic) that the competition data was synthetically generated from. Useful for target encoding priors (the original has slightly different class distribution: ~26.5% vs ~22.5% churn in train).

In [4]:
if os.getenv("KAGGLE_KERNEL_RUN_TYPE"):
    ENV = "kaggle"
    DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e3")
    ORIG_DIR = Path("/kaggle/input/datasets/cdeotte/s6e3-original-dataset")
else:
    ENV = "local"
    DATA_DIR = Path("./data")

print(f"Environment: {ENV}")

train = pd.read_csv(DATA_DIR / "train.csv").drop("id", axis=1)
test  = pd.read_csv(DATA_DIR / "test.csv").drop("id", axis=1)
orig_fn = "original.csv" if ENV == "local" else "WA_Fn-UseC_-Telco-Customer-Churn.csv"
original = pd.read_csv(ORIG_DIR / orig_fn).drop("customerID", axis=1)
sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
print("Data Loaded")

Environment: kaggle


Data Loaded


In [5]:
train.head(3)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.1,1653.85,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.5,3778.20,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.4,5841.35,No


Every transformation below is applied to train, test, and original datasets. 

- Map `{"No": 0, "Yes": 1}` the target variable `Churn`.
- Convert `SeniorCitizen` to string to treat it uniformly as a categorical.
- Fix `TotalCharges` in the original dataset.
- Remove redundant categories

In [6]:
TARGET = "Churn"

# Encode target
for df in (train, original):
    df[TARGET] = df[TARGET].map({"No": 0, "Yes": 1})

# SeniorCitizen to string (uniform categorical treatment)
for df in (train, test, original):
    df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# Fix original TotalCharges
original["TotalCharges"] = pd.to_numeric(original["TotalCharges"], errors="coerce")
original.loc[original["tenure"] == 0, "TotalCharges"] = 0

# Collapse redundant categories
internet_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
for col in internet_cols:
    for df in (train, test, original):
        df[col] = df[col].replace("No internet service", "No")

for df in (train, test, original):
    df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

FEATS = test.columns.tolist()
CATS  = test.select_dtypes(include="object").columns.tolist()
NUMS  = test.select_dtypes(include="number").columns.tolist()

print(f"Features: {len(FEATS)} total  |  {len(NUMS)} numeric  |  {len(CATS)} categorical")
print(f"Train: {train.shape}  |  Test: {test.shape}  |  Original: {original.shape}")
print(f"NUMS: {NUMS}")
print(f"CATS: {CATS}")

Features: 19 total  |  3 numeric  |  16 categorical
Train: (594194, 20)  |  Test: (254655, 19)  |  Original: (7043, 20)
NUMS: ['tenure', 'MonthlyCharges', 'TotalCharges']
CATS: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [7]:
# Engineered features `is_auto_paymentis_auto_payment` & `ChargeRes`
for df in (train, test, original):
    df["is_auto_payment"] = df["PaymentMethod"].isin(
        ["Credit card (automatic)", "Bank transfer (automatic)"]
    ).map({True: "Yes", False: "No"})
    df["ChargeRes"] = df["TotalCharges"] - df["tenure"] * df["MonthlyCharges"]

In [8]:
# Ratio features and service counts

INTERNET_SVC_COLS = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
for df in (train, test, original):
    df["AvgChargePerMonth"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["EffectiveTenure"]   = df["TotalCharges"] / (df["MonthlyCharges"] + 1e-4)
    df["ChargeResRate"]     = df["ChargeRes"] / (df["tenure"] + 1)
    df["n_services"]        = df[INTERNET_SVC_COLS].eq("Yes").sum(axis=1)
    df["ChargePerService"]  = df["MonthlyCharges"] / (df["n_services"] + 1)
    df["n_services"] = df["n_services"].astype(str)

FE_NUMS = ["AvgChargePerMonth"] # new num -> gets splined
FE_CATS = ["is_auto_payment", "n_services"] # new cats -> gets OHE/TE
LINEAR_DERIVED = ["ChargeRes", "EffectiveTenure", "ChargeResRate", "ChargePerService"]

FEATS = test.columns.tolist()
CATS  = test.select_dtypes(include="object").columns.tolist()
NUMS  = test.select_dtypes(include="number").columns.tolist()

print(f"NUMS ({len(NUMS)}): {NUMS}")
print(f"CATS ({len(CATS)}): {CATS}")
print(f"  of which NEW FE_NUMS: {FE_NUMS}")
print(f"  of which NEW FE_CATS: {FE_CATS}")
print(f"  LINEAR_DERIVED (no spline): {LINEAR_DERIVED}")

NUMS (8): ['tenure', 'MonthlyCharges', 'TotalCharges', 'ChargeRes', 'AvgChargePerMonth', 'EffectiveTenure', 'ChargeResRate', 'ChargePerService']
CATS (18): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'is_auto_payment', 'n_services']
  of which NEW FE_NUMS: ['AvgChargePerMonth']
  of which NEW FE_CATS: ['is_auto_payment', 'n_services']
  LINEAR_DERIVED (no spline): ['ChargeRes', 'EffectiveTenure', 'ChargeResRate', 'ChargePerService']


## Training LR

`train_linear()` is the central function that runs any preprocessor + model combination through stratified k-fold cross-validation. The function produces:
- OOF predictions (`oof`): predictions made when that row was in the validation fold.
- Test predictions (`test_preds`): average of predictions made by each fold's model on the test set. Averaging across folds reduces variance.
- Metadata (`meta`): CV AUC, per-fold scores, standard error, time, feature count.

Key features of the `train_linear()`:
- The preprocessor is freshly instantiated and `fit_transform` on each training fold to prevent data leakage.
- As we said, each fold's predicts of the full test set, and we average. This is equivalent to a 5 model ensemble for the test set.
- We calculate standard errors. When comparing two experiments, the meaningfull threshold for "real improvement" is roughly `SE_diff = sqrt(2) x SE`. Gains smaller than this are likely noise.
- OOF and TEST PREDS are save as `npy` files (when not in `TEST_MODE`) for ensemble building

In [9]:
if TEST_MODE:
    train_use =  train.sample(n=min(TEST_SAMPLE_N, len(train)), random_state=DSEED, ignore_index=True)
    test_use  = test.sample(n=min(TEST_SAMPLE_N, len(test)), random_state=DSEED, ignore_index=True)
    FOLDS = TEST_FOLDS
    print(f" TEST_MODE: train={len(train_use)}, test={len(test_use)}, folds={FOLDS}")
else:
    train_use = train
    test_use  = test
    FOLDS = FOLDS_FULL

X = train_use.drop(TARGET, axis=1)
y = train_use[TARGET].values

 TEST_MODE: train=200, test=200, folds=3


In [10]:
SAVE_DIR = Path(SAVE_DIR_NAME)
SAVE_DIR.mkdir(exist_ok=True)
KF = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=DSEED)

LOG = []

def train_linear(
    X, y, test_df, prep_fn, model,
    exp_name="logistic_reg",
    num_feats=None, cat_feats=None,
):
    if num_feats is None:
        num_feats = NUMS
    if cat_feats is None:
        cat_feats = CATS

    print("=" * 78)
    print(f" {exp_name}")
    print("=" * 78)

    feats = num_feats + cat_feats
    n_train = len(X)
    oof = np.zeros(n_train)
    test_preds = np.zeros(len(test_df))
    fold_auc = []
    n_cols = "?"
    t0 = time.time()

    for f, (tr_idx, va_idx) in enumerate(KF.split(X, y)):
        X_tr     = X.iloc[tr_idx][feats].copy()
        X_va     = X.iloc[va_idx][feats].copy()
        X_test   = test_df[feats].copy()
        y_tr     = y[tr_idx]
        y_va     = y[va_idx]

        prep = prep_fn(num_feats, cat_feats)
        X_tr_t  = prep.fit_transform(X_tr, y_tr)
        X_va_t  = prep.transform(X_va)
        X_te_t  = prep.transform(X_test)

        if f == 0:
            n_cols = X_tr_t.shape[1] if hasattr(X_tr_t, "shape") else "?"
            print(f"    Features in  -> {X_tr.shape[1]}   |   Features out -> {n_cols}")

        m = clone(model)

        # cuML 
        if HAS_CUML and isinstance(m, cuml.Base):
            import cupy as cp
            m.fit(cp.asarray(X_tr_t, dtype=cp.float32),
                  cp.asarray(y_tr,   dtype=cp.float32))
            va_pred = m.predict_proba(cp.asarray(X_va_t, dtype=cp.float32))[:, 1].get()
            te_pred = m.predict_proba(cp.asarray(X_te_t, dtype=cp.float32))[:, 1].get()
        else:
            m.fit(X_tr_t, y_tr)
            if hasattr(m, "predict_proba"):
                va_pred = m.predict_proba(X_va_t)[:, 1]
                te_pred = m.predict_proba(X_te_t)[:, 1]
            elif hasattr(m, "decision_function"):
                va_pred = m.decision_function(X_va_t)
                te_pred = m.decision_function(X_te_t)

        oof[va_idx] = va_pred
        test_preds += te_pred / FOLDS

        auc = roc_auc_score(y_va, va_pred)
        fold_auc.append(auc)
        print(f"    Fold {f+1}: AUC={auc:.6f}")

    cv_auc  = roc_auc_score(y, oof)
    elapsed = time.time() - t0
    se      = np.std(fold_auc) / np.sqrt(len(fold_auc))
    print(f"  CV AUC: {cv_auc:.6f}  |  mean±se: {np.mean(fold_auc):.6f} ± {se:.6f}  |  {elapsed:.1f}s\n")

    
    np.save(SAVE_DIR / f"{exp_name}_oof.npy",  oof)
    np.save(SAVE_DIR / f"{exp_name}_test.npy", test_preds)

    meta = {
        "experiment_name": exp_name,
        "cv_auc": cv_auc,
        "fold_scores": fold_auc,
        "se": se,
        "elapsed": elapsed,
        "n_features_out": n_cols,
        "test_mode": TEST_MODE,
    }
    return oof, test_preds, meta


## Preprocessor blocks - Helpers


Reusable transformer helpers that the preprocessor factories compose into full `ColumnTransformer` pipelines.

**Convention**: every encoder that outputs continuous values is followed by `StandardScaler` — this ensures L2/L1 regularization penalizes all features fairly regardless of their raw scale.

| Helper | Library | Type | What it does |
|--------|---------|------|-------------|
| `_ohe` | sklearn | Unsupervised | One-hot encoding, drops first level to avoid collinearity |
| `_ordinal` | sklearn | Unsupervised | Maps Contract to 0/1/2 preserving churn-rate monotonicity |
| `_te` | sklearn | Supervised | Smoothed target mean per category (smooth=20) |
| `_spline` | sklearn | Unsupervised | B-spline basis expansion for nonlinear numerical effects |
| `_woe` | category_encoders | Supervised | Weight of Evidence — output in log-odds space, native to LR |
| `_james_stein` | category_encoders | Supervised | Shrinkage TE with data-driven regularization |
| `_catboost_enc` | category_encoders | Supervised | Ordered TE — each row sees only prior observations |
| `_loo` | category_encoders | Supervised | Leave-one-out TE — aggressive, sigma noise for regularization |
| `_binary_enc` | category_encoders | Unsupervised | Ordinal → binary digits, log₂(k) columns per feature |
| `_helmert` | category_encoders | Unsupervised | Orthogonal contrasts — each level vs mean of subsequent levels |
| `_freq_enc` | feature_engine | Unsupervised | Category → proportion of rows (rarity signal) |
| `_eq_freq_disc` | feature_engine | Unsupervised | Quantile binning → OHE (piecewise-constant, different from splines) |
| `_tree_disc` | feature_engine | Supervised | Decision-tree-optimized binning → OHE |

Supervised helpers use the target during fit — leakage is prevented because `prep.fit_transform()` is called only on the training fold inside `train_linear()`.

In [11]:
CONTRACT_ORDER = ["Two year", "One year", "Month-to-month"]

# scikit-learn helpers

def _ohe(cat_feats):
    return OneHotEncoder(
        drop="first", sparse_output=False, handle_unknown="infrequent_if_exist"
    )

def _ordinal(cat_feats):
    return OrdinalEncoder(categories=[CONTRACT_ORDER])

def _te(cat_feats):
    return Pipeline([
        ("te",  TargetEncoder(smooth=20, random_state=DSEED)),
        ("sc",  StandardScaler()),
    ])

def _spline(n_knots=12):
    return Pipeline([
        ("spline", SplineTransformer(
            n_knots=n_knots, degree=3, knots="quantile", include_bias=False
        )),
        ("sc",     StandardScaler())
    ])

# category encoders helpers
if HAS_CE:
    def _woe(cat_feats):
        """Weight of Evidence: maps to ln(event_rate / non_event_rate).
        Natural fit for LR — output is already in log-odds space."""
        return Pipeline([
            ("woe", ce.WOEEncoder(cols=cat_feats, random_state=DSEED, randomized=True, sigma=0.05)),
            ("sc",  StandardScaler()),
        ])

    def _james_stein(cat_feats):
        """James-Stein shrinkage encoder: data-driven shrinkage factor,
        different regularization from sklearn TE's smooth parameter."""
        return Pipeline([
            ("js", ce.JamesSteinEncoder(cols=cat_feats, random_state=DSEED)),
            ("sc", StandardScaler()),
        ])

    def _catboost_enc(cat_feats):
        """CatBoost-style ordered TE: each row encoded using only prior
        observations. Different bias structure from standard k-fold TE."""
        return Pipeline([
            ("cb", ce.CatBoostEncoder(cols=cat_feats, random_state=DSEED, sigma=0.05)),
            ("sc", StandardScaler()),
        ])

    def _loo(cat_feats):
        """Leave-One-Out TE: aggressive, high variance. sigma=0.05 adds
        Gaussian noise. Safe when fit only on training fold."""
        return Pipeline([
            ("loo", ce.LeaveOneOutEncoder(cols=cat_feats, random_state=DSEED, sigma=0.05)),
            ("sc",  StandardScaler()),
        ])

    def _binary_enc(cat_feats):
        """Binary encoder: log2(k) columns per feature (vs k-1 for OHE).
        Structural alternative — pure diversity play."""
        return Pipeline([
            ("bin", ce.BinaryEncoder(cols=cat_feats)),
            ("sc",  StandardScaler()),
        ])

    def _helmert(cat_feats):
        """Helmert contrast coding: each level vs mean of subsequent levels.
        Orthogonal contrasts — useful for ordinal categoricals."""
        return Pipeline([
            ("helm", ce.HelmertEncoder(cols=cat_feats)),
            ("sc",   StandardScaler()),
        ])

# feature engine helpers

if HAS_FE:
    def _freq_enc(cat_feats):
        """Frequency encoding: category -> proportion of rows with that value.
        Unsupervised — no target leakage."""
        return Pipeline([
            ("freq", CountFrequencyEncoder(
                encoding_method="frequency",
                variables=cat_feats,
            )),
            ("sc", StandardScaler()),
        ])

    def _eq_freq_disc(num_feats, n_bins=10):
        """Equal-frequency (quantile) binning -> OHE of bins.
        Creates piecewise-constant features, structurally different from splines."""
        return Pipeline([
            ("disc", EqualFrequencyDiscretiser(
                q=n_bins,
                variables=num_feats,
                return_boundaries=True,
            )),
            ("ohe", OneHotEncoder(sparse_output=False, handle_unknown="infrequent_if_exist")),
        ])

    def _tree_disc(num_feats, max_depth=3):
        """Decision tree discretiser: supervised binning that finds optimal
        split points w.r.t. the target. max_depth=3 -> up to 8 bins."""
        return Pipeline([
            ("disc", DecisionTreeDiscretiser(
                variables=num_feats,
                regression=False,
                scoring="roc_auc",
                cv=3,
                random_state=DSEED,
                param_grid={"max_depth": [max_depth]},
            )),
            ("ohe", OneHotEncoder(sparse_output=False, handle_unknown="infrequent_if_exist")),
        ])

## Preprocessor factories

Every factory follows the same logic: `make_X_prep(num_feats, cat_feats) -> ColumnTransformer`.

They are organized into groups. Before each group of experiments, we display the ColumnTransformer diagram so you can see exactly what transformation are applied to which columns.

### Group A: Baselines

These establish the performance ladder. Each adds one concept over the previous, so any AUC gain is attributable to that specific addition.

| Factory | Numericals | Categoricals | What it tests |
|---------|-----------|-------------|--------------|
| `make_basic1_prep` | Scale | OHE | Simplest possible LR baseline |
| `make_basic2_prep` | Scale | OHE + Ordinal(Contract) | Does exploiting Contract's monotonic churn ordering help? |
| `make_te_cat1_prep` | Scale | TE | Does replacing OHE with target means help? |
| `make_te_ohe_prep` | Scale | OHE + Ordinal + TE | Does *dual* encoding help? OHE captures jumps between categories, TE captures magnitude |
| `make_spline_prep` | Spline | OHE | Does modeling nonlinear numerical effects (e.g. tenure vs churn curve) help? |
| `make_spline_te_prep` | Spline(base) + Scale(derived) | OHE + Ordinal + TE | Best baseline: nonlinear nums, derived features kept linear to avoid redundancy with splined sources |


In [12]:
# Group A: Baselines (scale/encode only, no interactions)

# Basic (scale nums + OHE cats)
def make_basic1_prep(num_feats, cat_feats):
    return ColumnTransformer([
        ("num",  StandardScaler(), num_feats),
        ("cat",  _ohe(cat_feats),  cat_feats),
    ])

# Ordinal Contract + OHE rest
def make_basic2_prep(num_feats, cat_feats):
    ohe_feats = [f for f in cat_feats if f != "Contract"]
    return ColumnTransformer([
        ("num",     StandardScaler(), num_feats),
        ("ordinal", _ordinal(cat_feats), ["Contract"]),
        ("cat",     _ohe(ohe_feats), ohe_feats),
    ])

# Target-encoded cats only
def make_te_cat1_prep(num_feats, cat_feats):
    return ColumnTransformer([
        ("num",    StandardScaler(), num_feats),
        ("te_cat", _te(cat_feats),   cat_feats),
    ])

# OHE + TE dual encoding
def make_te_ohe_prep(num_feats, cat_feats):
    ohe_feats = [f for f in cat_feats if f != "Contract"]
    return ColumnTransformer([
        ("num",     StandardScaler(), num_feats),
        ("ordinal", _ordinal(cat_feats), ["Contract"]),
        ("ohe",     _ohe(ohe_feats), ohe_feats),
        ("te_cat",  _te(cat_feats),  cat_feats),
    ])

# Spline numericals + OHE
def make_spline_prep(num_feats, cat_feats, n_knots=10):
    return ColumnTransformer([
        ("spl", _spline(n_knots), num_feats),
        ("cat", _ohe(cat_feats),  cat_feats),
    ])

# Spline (excl LINEAR_DERIVED) + LINEAR_DERIVED + OHE + Ordinal contract + TE
def make_spline_te_prep(num_feats, cat_feats, n_knots=10):
    ohe_feats = [f for f in cat_feats if f != "Contract"]
    spline_nums = [f for f in num_feats if f not in LINEAR_DERIVED]
    linear_nums = [f for f in num_feats if f in LINEAR_DERIVED]
    return ColumnTransformer([
        ("spl",     _spline(n_knots),  spline_nums),
        ("lin",     StandardScaler(),   linear_nums),
        ("ohe",     _ohe(ohe_feats),    ohe_feats),
        ("ordinal", _ordinal(cat_feats), ["Contract"]),
        ("te_cat",  _te(cat_feats),     cat_feats),
    ])

## Group B: Interactions & polynomials

LR can only learn linear relationships in the feature space it receives. These preprocessors explicitly construct the nonlinear terms and cross-feature products that LR cannot discover on its own.

| Factory | Strategy | Key idea | 
|---------|----------|----------|
| `make_spline_interact_prep` | Spline + OHE + TE + spline×OHE | Separate nonlinear tenure/charge curves per category (e.g. tenure×Contract) — the single highest-value addition for churn |
| `make_poly_prep` | Degree-2 poly on key nums + OHE + TE | tenure², MonthlyCharges², tenure×MonthlyCharges on selected features |
| `make_poly2_prep` | Degree-2 poly on ALL nums + OHE + TE | All pairwise products and squares — brute force, relies on regularization to prune | 
| `make_num_interact_prep` | Scale + OHE + TE + pairwise products | Only cross-products (no squares) on selected nums — lightweight interaction variant |
| `make_sink_prep` | Spline + linear + OHE + TE + spline×OHE + poly | Everything combined — max feature count, needs L1/ElasticNet |

**`LINEAR_DERIVED` separation**: derived features (ChargeRes, EffectiveTenure, etc.) get linear scaling, not splines, to avoid redundancy with their already-splined source features.

### Polynomial transformation (scikit-learn)
| Setting | Output |
|---|---|
| `interaction_only=False, include_bias=False` | a, b, c, a², b², c², a×b, a×c, b×c |
| `interaction_only=True, include_bias=False` | a, b, c, a×b, a×c, b×c |

In [13]:
# Group B: Interactions & polynomials

# Spline + lin ("ChargeRes", "ChargePerService", "ChargeResRate") +  interact_block (Spline×OHE) 

def make_spline_interact_prep(num_feats, cat_feats, n_knots=10,
                               interact_nums=None, interact_cats=None):
    if interact_nums is None:
        interact_nums = [f for f in ["tenure", "MonthlyCharges"] if f in num_feats]
    if interact_cats is None:
        interact_cats = cat_feats

    spline_nums = [f for f in num_feats if f not in LINEAR_DERIVED]
    linear_nums = [f for f in num_feats if f in LINEAR_DERIVED]

    # Interaction block: spline(interact_nums) × OHE(interact_cats)
    interact_block = Pipeline([
        ("ct", ColumnTransformer([
            ("spl_i", _spline(n_knots), interact_nums),
            ("ohe_i", _ohe(interact_cats), interact_cats),
        ])),
        ("poly", PolynomialFeatures(
            degree=2, interaction_only=True, include_bias=False
        )),
        ("sc",   StandardScaler())
    ])

    return ColumnTransformer([
        ("spl",      _spline(n_knots), spline_nums),
        ("lin",      StandardScaler(),  linear_nums),
        ("ohe",      _ohe(cat_feats),   cat_feats),
        ("te_cat",   _te(cat_feats),    cat_feats),
        ("interact", interact_block,    interact_nums + interact_cats),
    ])


# Degree-2 polynomial on key numericals
def make_poly_prep(num_feats, cat_feats, poly_feats=None, degree=2):
    if poly_feats is None:
        poly_feats = [f for f in ["tenure", "MonthlyCharges", "TotalCharges"]
                      if f in num_feats]
    other_nums = [f for f in num_feats if f not in poly_feats]

    poly_block = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("sc",   StandardScaler())
    ])

    return ColumnTransformer([
        ("poly",    poly_block,       poly_feats),
        ("num",     StandardScaler(), other_nums),
        ("ohe",     _ohe(cat_feats),  cat_feats),
        ("te_cat",  _te(cat_feats),   cat_feats),
    ])

# Degree-2 polynomial on all numericals
def make_poly2_prep(num_feats, cat_feats, poly_feats=None, degree=2):
    poly_block = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("sc",   StandardScaler())
    ])

    return ColumnTransformer([
        ("poly",    poly_block,       num_feats),
        ("ohe",     _ohe(cat_feats),  cat_feats),
        ("te_cat",  _te(cat_feats),   cat_feats),
    ])


# Num × Num interactions (scaled products)
def make_num_interact_prep(num_feats, cat_feats, pair_feats=None):
    if pair_feats is None:
        pair_feats = [f for f in ["tenure", "MonthlyCharges", "ChargeRes"]
                      if f in num_feats]
    other_nums = [f for f in num_feats if f not in pair_feats]
    return ColumnTransformer([
        ("num",       StandardScaler(), other_nums),
        ("ohe",       _ohe(cat_feats),  cat_feats),
        ("te_cat",    _te(cat_feats),   cat_feats),
        ("num_pairs", Pipeline([
            ("poly", PolynomialFeatures(degree=2, interaction_only=True,
                                         include_bias=False)),
             ("sc",   StandardScaler())
        ]), pair_feats),
    ])


# spline + linear + OHE + TE + (splinexOHE) + (polyNums2)
def make_sink_prep(num_feats, cat_feats, n_knots=10):
    spline_nums   = [f for f in num_feats if f not in LINEAR_DERIVED]
    linear_nums   = [f for f in num_feats if f in LINEAR_DERIVED]
    interact_nums = [f for f in ["tenure", "MonthlyCharges"] if f in num_feats]
    poly_nums     = [f for f in ["tenure", "MonthlyCharges", "TotalCharges"]
                     if f in num_feats]

    interact_block = Pipeline([
        ("ct", ColumnTransformer([
            ("spl_i", _spline(n_knots), interact_nums),
            ("ohe_i", _ohe(cat_feats), cat_feats),
        ])),
        ("poly", PolynomialFeatures(
            degree=2, interaction_only=True, include_bias=False
        )),
        ("se", StandardScaler())
    ])

    poly_block = Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("sc",   StandardScaler())
    ])

    return ColumnTransformer([
        ("spl",      _spline(n_knots), spline_nums),
        ("lin",      StandardScaler(),  linear_nums),
        ("ohe",      _ohe(cat_feats),   cat_feats),
        ("te_cat",   _te(cat_feats),    cat_feats),
        ("interact", interact_block,    interact_nums + cat_feats),
        ("poly",     poly_block,        poly_nums),
    ])


## Group C: Alternative encoders (category_encoders)

All share the same numerical preprocessing — spline(base nums) + linear(derived) — and only vary how categoricals are encoded. Any AUC difference between two variants is attributable entirely to the encoding strategy.

`_ce_spline_prep` is a template that avoids repeating the same ColumnTransformer structure 7 times — each variant is a one-liner passing its specific encoder.

| Factory | Encoder | Paired with | Why it adds diversity |
|---------|---------|------------|----------------------|
| `make_woe_spline_prep` | WoE | OHE | Log-odds space — most natural encoding for LR |
| `make_js_spline_prep` | James-Stein | OHE | Data-driven shrinkage (vs sklearn TE's fixed smooth) |
| `make_cb_spline_prep` | CatBoost | OHE | Ordered TE — different bias from k-fold TE |
| `make_loo_spline_prep` | LOO | OHE | Most aggressive TE — high variance, different errors |
| `make_binary_spline_prep` | Binary | TE | Not target-based at all — structurally orthogonal |
| `make_helmert_spline_prep` | Helmert | TE | Orthogonal contrasts — different from any mean-based encoder |
| `make_multi_enc_spline_prep` | OHE + WoE + JS + TE | — | Stacks 4 encodings of the same cats — max category diversity, relies on regularization to blend |

In [14]:
# Group C: Alternative Encoders (category encoders)
# All share the following architecture: spline(base_nums) + linear(derived) + encoder cats

if HAS_CE:
    # helper
    def _ce_spline_prep(encoder_fn, num_feats, cat_feats, n_knots=12):
        spline_nums = [f for f in num_feats if f not in LINEAR_DERIVED]
        linear_nums = [f for f in num_feats if f in LINEAR_DERIVED]
        return ColumnTransformer([
            ("spl", _spline(n_knots), spline_nums),
            ("lin", StandardScaler(),  linear_nums),
            ("ohe", _ohe(cat_feats),   cat_feats),
            ("enc", encoder_fn(cat_feats), cat_feats),
        ])

    def make_woe_spline_prep(num_feats, cat_feats, **kw):
        return _ce_spline_prep(_woe, num_feats, cat_feats, **kw)
    def make_js_spline_prep(num_feats, cat_feats, **kw):
        return _ce_spline_prep(_james_stein, num_feats, cat_feats, **kw)
    def make_cb_spline_prep(num_feats, cat_feats, **kw):
        return _ce_spline_prep(_catboost_enc, num_feats, cat_feats, **kw)
    def make_loo_spline_prep(num_feats, cat_feats, **kw):
        return _ce_spline_prep(_loo, num_feats, cat_feats, **kw)
    def make_binary_spline_prep(num_feats, cat_feats, n_knots=10):
        spline_nums = [f for f in num_feats if f not in LINEAR_DERIVED]
        linear_nums = [f for f in num_feats if f in LINEAR_DERIVED]
        return ColumnTransformer([
            ("spl", _spline(n_knots), spline_nums),
            ("lin", StandardScaler(),  linear_nums),
            ("bin", _binary_enc(cat_feats), cat_feats),
            ("te",  _te(cat_feats),    cat_feats),
        ])

    def make_helmert_spline_prep(num_feats, cat_feats, n_knots=10):
        spline_nums = [f for f in num_feats if f not in LINEAR_DERIVED]
        linear_nums = [f for f in num_feats if f in LINEAR_DERIVED]
        return ColumnTransformer([
            ("spl",  _spline(n_knots), spline_nums),
            ("lin",  StandardScaler(),  linear_nums),
            ("helm", _helmert(cat_feats), cat_feats),
            ("te",   _te(cat_feats),    cat_feats),
        ])
    
    def make_multi_enc_spline_prep(num_feats, cat_feats, n_knots=10):
        spline_nums = [f for f in num_feats if f not in LINEAR_DERIVED]
        linear_nums = [f for f in num_feats if f in LINEAR_DERIVED]
        return ColumnTransformer([
            ("spl", _spline(n_knots), spline_nums),
            ("lin", StandardScaler(),  linear_nums),
            ("ohe", _ohe(cat_feats),   cat_feats),
            ("woe", _woe(cat_feats),   cat_feats),
            ("js",  _james_stein(cat_feats), cat_feats),
            ("te",  _te(cat_feats),    cat_feats),
        ])

## Group D: feature_engine

These preprocessors transform the feature space structurally — binning continuous features into discrete intervals or replacing OHE with frequency counts. This creates fundamentally different representations from the spline-based approaches in Groups A–B.

| Factory | Numericals | Categoricals | How it differs |
|---------|-----------|-------------|---------------|
| `make_freq_spline_prep` | Spline + linear | OHE + Freq | Frequency encoding captures rarity signal — completely unsupervised, zero leakage risk |
| `make_freq_te_spline_prep` | Spline + linear | OHE + TE + Freq | Triple encoding: categorical jumps (OHE) + target signal (TE) + rarity signal (Freq) |
| `make_binned_te_prep` | Quantile bins(3 key) + Scale(rest) | OHE + TE | Piecewise-constant step functions instead of smooth splines — different bias structure |
| `make_binned_freq_te_prep` | Quantile bins(3 key) + Scale(rest) | OHE + Freq + TE | Binned nums + triple cat encoding — max diversity variant |
| `make_tree_disc_prep` | Tree-optimized bins(3 key) + Scale(rest) | OHE + TE | Supervised binning — bin boundaries chosen to maximize AUC, not equal-frequency |

In [15]:
# Group D: feature_engine preprocessors
if HAS_FE:
    def make_freq_spline_prep(num_feats, cat_feats, n_knots=10):
        spline_nums = [f for f in num_feats if f not in LINEAR_DERIVED]
        linear_nums = [f for f in num_feats if f in LINEAR_DERIVED]
        return ColumnTransformer([
            ("spl",  _spline(n_knots), spline_nums),
            ("lin",  StandardScaler(),  linear_nums),
            ("ohe",  _ohe(cat_feats),   cat_feats),
            ("freq", _freq_enc(cat_feats), cat_feats),
        ])

    def make_freq_te_spline_prep(num_feats, cat_feats, n_knots=10):
        spline_nums = [f for f in num_feats if f not in LINEAR_DERIVED]
        linear_nums = [f for f in num_feats if f in LINEAR_DERIVED]
        return ColumnTransformer([
            ("spl",  _spline(n_knots), spline_nums),
            ("lin",  StandardScaler(),  linear_nums),
            ("ohe",  _ohe(cat_feats),   cat_feats),
            ("te", _te(cat_feats), cat_feats),
            ("freq", _freq_enc(cat_feats), cat_feats),
        ])

    def make_binned_te_prep(num_feats, cat_feats, n_bins=10):
        bin_nums   = [f for f in ["tenure", "MonthlyCharges", "TotalCharges"] if f in num_feats]
        other_nums = [f for f in num_feats if f not in bin_nums]
        return ColumnTransformer([
            ("binned", _eq_freq_disc(bin_nums, n_bins=n_bins), bin_nums),
            ("num",    StandardScaler(), other_nums),
            ("ohe",    _ohe(cat_feats),  cat_feats),
            ("te",     _te(cat_feats),   cat_feats)
        ])
    def make_binned_freq_te_prep(num_feats, cat_feats, n_bins=10):
        bin_nums   = [f for f in ["tenure", "MonthlyCharges", "TotalCharges"] if f in num_feats]
        other_nums = [f for f in num_feats if f not in bin_nums]
        return ColumnTransformer([
            ("binned", _eq_freq_disc(bin_nums, n_bins=n_bins), bin_nums),
            ("num",    StandardScaler(), other_nums),
            ("ohe",    _ohe(cat_feats),  cat_feats),
            ("freq", _freq_enc(cat_feats), cat_feats),
            ("te",     _te(cat_feats),   cat_feats),
        ])


    def make_tree_disc_prep(num_feats, cat_feats, max_depth=3):
        disc_nums  = [f for f in ["tenure", "MonthlyCharges", "TotalCharges"] if f in num_feats]
        other_nums = [f for f in num_feats if f not in disc_nums]
        return ColumnTransformer([
            ("tree_disc", _tree_disc(disc_nums, max_depth=max_depth), disc_nums),
            ("num",       StandardScaler(), other_nums),
            ("ohe",       _ohe(cat_feats),  cat_feats),
            ("te",        _te(cat_feats),   cat_feats),
        ])

## Model Factory

`make_lr` creates logistic regression models across three penalty types and two backends (CPU/GPU)

In [16]:
def make_lr(C=1.0, penalty="l2", max_iter=50000, solver=None, l1_ratio=None):
    """LR with auto solver selection based on penalty."""
    if penalty == "elasticnet" and l1_ratio is None:
        l1_ratio = 0.5
    # cuML path
    if USE_GPU:
        if penalty == "elasticnet":
            print("    [note] cuML has no elasticnet —  falling back to sklearn saga")
        else:
            return cuLogisticRegression(
                C=C, penalty=penalty, max_iter=max_iter, solver="qn", tol=1e-3
            )
    # sklearn path — auto-select solver
    if solver is None:
        if penalty == "l2":
            solver = "lbfgs"
        elif penalty == "l1":
            solver = "saga"
        elif penalty == "elasticnet":
            solver = "saga"
        else:
            solver = "lbfgs"

    kwargs = dict(C=C, penalty=penalty, max_iter=max_iter, solver=solver,
                  random_state=DSEED, n_jobs=-1)
    if penalty == "elasticnet":
        kwargs["l1_ratio"] = l1_ratio
    return LogisticRegression(**kwargs)


def make_lr_l1(C=0.1, max_iter=50000):
    """Convenience: L1-regularized LR. Default C=0.1 (stronger reg for high-dim)."""
    return make_lr(C=C, penalty="l1", max_iter=max_iter)


def make_lr_enet(C=0.1, l1_ratio=0.5, max_iter=50000):
    """Convenience: ElasticNet LR. Best for high-dim with grouped collinearity."""
    return make_lr(C=C, penalty="elasticnet", l1_ratio=l1_ratio, max_iter=max_iter)

## Baseline preprocessors

We run 6 baseline preprocessors with L2 regularization (C=0.1) to establish the performance ladder.

In [17]:
make_basic1_prep(NUMS, CATS)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'ChargeRes', 'AvgChargePerMonth',
                                  'EffectiveTenure', 'ChargeResRate',
                                  'ChargePerService']),
                                ('cat',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='infrequent_if_exist',
                                               sparse_output=False),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [18]:
lr = make_lr()
oof_1, tp_1, m1 = train_linear(X, y, test_use, make_basic1_prep, lr, exp_name="lr_basic1")
LOG.append(m1)

 lr_basic1
    Features in  -> 26   |   Features out -> 35


    Fold 1: AUC=0.883529
    Fold 2: AUC=0.772109
    Fold 3: AUC=0.829532
  CV AUC: 0.829522  |  mean±se: 0.828390 ± 0.026266  |  3.6s



In [19]:
make_basic2_prep(NUMS, CATS)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'ChargeRes', 'AvgChargePerMonth',
                                  'EffectiveTenure', 'ChargeResRate',
                                  'ChargePerService']),
                                ('ordinal',
                                 OrdinalEncoder(categories=[['Two year',
                                                             'One year',
                                                             'Month-to-month']]),
                                 ['Contract']),
                                ('cat',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='infrequent_if_exist',
                                               sparse_output=False),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'PaperlessBilling',
                                  'PaymentMethod', 'is_auto_payment',
                                  'n_services'])])

In [20]:
oof_2, tp_2, m2 = train_linear(X, y, test_use, make_basic2_prep, lr, exp_name="lr_basic2_ord")
LOG.append(m2)

 lr_basic2_ord


    Features in  -> 26   |   Features out -> 34
    Fold 1: AUC=0.881176
    Fold 2: AUC=0.770975
    Fold 3: AUC=0.833133
  CV AUC: 0.831341  |  mean±se: 0.828428 ± 0.026046  |  0.2s



In [21]:
make_te_cat1_prep(NUMS, CATS)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'ChargeRes', 'AvgChargePerMonth',
                                  'EffectiveTenure', 'ChargeResRate',
                                  'ChargePerService']),
                                ('te_cat',
                                 Pipeline(steps=[('te',
                                                  TargetEncoder(random_state=10431053,
                                                                smooth=20)),
                                                 ('sc', StandardScaler())]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [22]:
oof_3, tp_3, m3 = train_linear(X, y, test_use, make_te_cat1_prep, lr, exp_name="lr_te_only")
LOG.append(m3)

 lr_te_only
    Features in  -> 26   |   Features out -> 26
    Fold 1: AUC=0.838824
    Fold 2: AUC=0.710884
    Fold 3: AUC=0.817527
  CV AUC: 0.794309  |  mean±se: 0.789078 ± 0.032315  |  0.2s



In [23]:
make_te_ohe_prep(NUMS, CATS)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'ChargeRes', 'AvgChargePerMonth',
                                  'EffectiveTenure', 'ChargeResRate',
                                  'ChargePerService']),
                                ('ordinal',
                                 OrdinalEncoder(categories=[['Two year',
                                                             'One year',
                                                             'Month-to-month']]),
                                 ['Contract']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='infrequent_if_exist',
                                               s...
                                 Pipeline(steps=[('te',
                                                  TargetEncoder(random_state=10431053,
                                                                smooth=20)),
                                                 ('sc', StandardScaler())]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [24]:
oof_4, tp_4, m4 = train_linear(X, y, test_use, make_te_ohe_prep, lr, exp_name="lr_te_ohe")
LOG.append(m4)

 lr_te_ohe
    Features in  -> 26   |   Features out -> 52
    Fold 1: AUC=0.836471
    Fold 2: AUC=0.725624


    Fold 3: AUC=0.824730
  CV AUC: 0.799636  |  mean±se: 0.795608 ± 0.028705  |  0.3s



In [25]:
make_spline_prep(NUMS, CATS)

ColumnTransformer(transformers=[('spl',
                                 Pipeline(steps=[('spline',
                                                  SplineTransformer(include_bias=False,
                                                                    knots='quantile',
                                                                    n_knots=10)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'ChargeRes', 'AvgChargePerMonth',
                                  'EffectiveTenure', 'ChargeResRate',
                                  'ChargePerService']),
                                ('cat',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='infrequent_if_exist',
                                               sparse_output=False),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [26]:
oof_5, tp_5, m5 = train_linear(X, y, test_use, make_spline_prep, lr, exp_name="lr_spline12")
LOG.append(m5)

 lr_spline12
    Features in  -> 26   |   Features out -> 115
    Fold 1: AUC=0.871765


    Fold 2: AUC=0.685941


    Fold 3: AUC=0.770708
  CV AUC: 0.778456  |  mean±se: 0.776138 ± 0.043855  |  0.2s



In [27]:
make_spline_te_prep(NUMS, CATS)

ColumnTransformer(transformers=[('spl',
                                 Pipeline(steps=[('spline',
                                                  SplineTransformer(include_bias=False,
                                                                    knots='quantile',
                                                                    n_knots=10)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'AvgChargePerMonth']),
                                ('lin', StandardScaler(),
                                 ['ChargeRes', 'EffectiveTenure',
                                  'ChargeResRate', 'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               ha...
                                 Pipeline(steps=[('te',
                                                  TargetEncoder(random_state=10431053,
                                                                smooth=20)),
                                                 ('sc', StandardScaler())]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [28]:
oof_6, tp_6, m6 = train_linear(X, y, test_use, make_spline_te_prep, lr, exp_name="lr_spline_te")
LOG.append(m6)

 lr_spline_te


    Features in  -> 26   |   Features out -> 92


    Fold 1: AUC=0.811765


    Fold 2: AUC=0.706349


    Fold 3: AUC=0.776711
  CV AUC: 0.768841  |  mean±se: 0.764942 ± 0.025307  |  0.4s



## Interactions and polynomials

In [29]:
make_spline_interact_prep(NUMS, CATS)

ColumnTransformer(transformers=[('spl',
                                 Pipeline(steps=[('spline',
                                                  SplineTransformer(include_bias=False,
                                                                    knots='quantile',
                                                                    n_knots=10)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'AvgChargePerMonth']),
                                ('lin', StandardScaler(),
                                 ['ChargeRes', 'EffectiveTenure',
                                  'ChargeResRate', 'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               ha...
                                                                     interaction_only=True)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'gender',
                                  'SeniorCitizen', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [30]:
# Spline × OHE interactions  — the big one
oof_7, tp_7, m7 = train_linear(
    X, y, test_use, make_spline_interact_prep, lr, exp_name="lr_spline_interact")
LOG.append(m7)

 lr_spline_interact
    Features in  -> 26   |   Features out -> 1318
    Fold 1: AUC=0.845882


    Fold 2: AUC=0.691610
    Fold 3: AUC=0.703481
  CV AUC: 0.752859  |  mean±se: 0.746991 ± 0.040469  |  0.5s



In [31]:
make_poly_prep(NUMS, CATS)

ColumnTransformer(transformers=[('poly',
                                 Pipeline(steps=[('poly',
                                                  PolynomialFeatures(include_bias=False)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('num', StandardScaler(),
                                 ['ChargeRes', 'AvgChargePerMonth',
                                  'EffectiveTenure', 'ChargeResRate',
                                  'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='infrequent_if_...
                                 Pipeline(steps=[('te',
                                                  TargetEncoder(random_state=10431053,
                                                                smooth=20)),
                                                 ('sc', StandardScaler())]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [32]:
# Degree-2 polynomials on key numericals
oof_8, tp_8, m8 = train_linear(
    X, y, test_use, make_poly_prep, lr, exp_name="lr_poly2")
LOG.append(m8)

 lr_poly2
    Features in  -> 26   |   Features out -> 59
    Fold 1: AUC=0.836471


    Fold 2: AUC=0.716553
    Fold 3: AUC=0.823529
  CV AUC: 0.798337  |  mean±se: 0.792184 ± 0.031027  |  0.3s



In [33]:
make_poly2_prep(NUMS, CATS)

ColumnTransformer(transformers=[('poly',
                                 Pipeline(steps=[('poly',
                                                  PolynomialFeatures(include_bias=False)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'ChargeRes', 'AvgChargePerMonth',
                                  'EffectiveTenure', 'ChargeResRate',
                                  'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='infrequent_if_exist',
                                               sparse_output=False)...
                                 Pipeline(steps=[('te',
                                                  TargetEncoder(random_state=10431053,
                                                                smooth=20)),
                                                 ('sc', StandardScaler())]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [34]:
oof_9, tp_9, m9 = train_linear(
    X, y, test_use, make_poly2_prep, lr, exp_name="lr_poly2_2")
LOG.append(m9)

 lr_poly2_2


    Features in  -> 26   |   Features out -> 89
    Fold 1: AUC=0.857647


    Fold 2: AUC=0.666667


    Fold 3: AUC=0.812725
  CV AUC: 0.781055  |  mean±se: 0.779013 ± 0.047071  |  0.3s



In [35]:
make_num_interact_prep(NUMS, CATS)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['TotalCharges', 'AvgChargePerMonth',
                                  'EffectiveTenure', 'ChargeResRate',
                                  'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='infrequent_if_exist',
                                               sparse_output=False),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecuri...
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services']),
                                ('num_pairs',
                                 Pipeline(steps=[('poly',
                                                  PolynomialFeatures(include_bias=False,
                                                                     interaction_only=True)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'ChargeRes'])])

In [36]:
# Num × Num interactions
oof_10, tp_10, m10 = train_linear(
    X, y, test_use, make_num_interact_prep, lr, exp_name="lr_num_interact")
LOG.append(m10)

 lr_num_interact
    Features in  -> 26   |   Features out -> 56


    Fold 1: AUC=0.841176


    Fold 2: AUC=0.714286
    Fold 3: AUC=0.809124
  CV AUC: 0.793399  |  mean±se: 0.788195 ± 0.031105  |  0.3s



In [37]:
make_sink_prep(NUMS, CATS)

ColumnTransformer(transformers=[('spl',
                                 Pipeline(steps=[('spline',
                                                  SplineTransformer(include_bias=False,
                                                                    knots='quantile',
                                                                    n_knots=10)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'AvgChargePerMonth']),
                                ('lin', StandardScaler(),
                                 ['ChargeRes', 'EffectiveTenure',
                                  'ChargeResRate', 'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               ha...
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services']),
                                ('poly',
                                 Pipeline(steps=[('poly',
                                                  PolynomialFeatures(include_bias=False)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges'])])

In [38]:
# Kitchen sink (everything combined)
oof_11, tp_11, m11 = train_linear(
    X, y, test_use, make_sink_prep, lr, exp_name="lr_sink")
LOG.append(m11)

 lr_sink
    Features in  -> 26   |   Features out -> 1327
    Fold 1: AUC=0.848235


    Fold 2: AUC=0.695011
    Fold 3: AUC=0.705882
  CV AUC: 0.753248  |  mean±se: 0.749710 ± 0.040304  |  0.5s



## Regularization 

The default C=1.0 may not be the optimal for the interaction preprocessor. We sweep `C` to find the bias-variance sweet spot on the high-dimensional preprocessors.

In [39]:
make_spline_interact_prep(NUMS, CATS)

ColumnTransformer(transformers=[('spl',
                                 Pipeline(steps=[('spline',
                                                  SplineTransformer(include_bias=False,
                                                                    knots='quantile',
                                                                    n_knots=10)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'AvgChargePerMonth']),
                                ('lin', StandardScaler(),
                                 ['ChargeRes', 'EffectiveTenure',
                                  'ChargeResRate', 'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               ha...
                                                                     interaction_only=True)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'gender',
                                  'SeniorCitizen', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [40]:
for C_val in [0.1, 1.0, 5.0]:
    lr_c = make_lr(C=C_val)
    _, _, mc = train_linear(
        X, y, test_use, make_spline_interact_prep, lr_c,
        exp_name=f"lr_interact_C{C_val}")
    LOG.append(mc)

 lr_interact_C0.1
    Features in  -> 26   |   Features out -> 1318
    Fold 1: AUC=0.871765


    Fold 2: AUC=0.713152
    Fold 3: AUC=0.728691
  CV AUC: 0.773649  |  mean±se: 0.771203 ± 0.041217  |  0.5s

 lr_interact_C1.0


    Features in  -> 26   |   Features out -> 1318
    Fold 1: AUC=0.845882
    Fold 2: AUC=0.690476


    Fold 3: AUC=0.704682
  CV AUC: 0.752859  |  mean±se: 0.747013 ± 0.040502  |  0.5s

 lr_interact_C5.0
    Features in  -> 26   |   Features out -> 1318
    Fold 1: AUC=0.836471


    Fold 2: AUC=0.683673
    Fold 3: AUC=0.692677
  CV AUC: 0.746752  |  mean±se: 0.737607 ± 0.040417  |  0.5s



## L1 and ElasticNet

For high-dimensional preprocessors, L1 and ElasticNet can help with redundant interaction terms that L2 only shrink. This can improve the model by reducing effective dimensionality. Also, L1 and L2 on the same preprocessor produce different coefficient vectos which it creates diversity. 

In [41]:
make_multi_enc_spline_prep(NUMS, CATS)

ColumnTransformer(transformers=[('spl',
                                 Pipeline(steps=[('spline',
                                                  SplineTransformer(include_bias=False,
                                                                    knots='quantile',
                                                                    n_knots=10)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'AvgChargePerMonth']),
                                ('lin', StandardScaler(),
                                 ['ChargeRes', 'EffectiveTenure',
                                  'ChargeResRate', 'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               ha...
                                 Pipeline(steps=[('te',
                                                  TargetEncoder(random_state=10431053,
                                                                smooth=20)),
                                                 ('sc', StandardScaler())]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services'])])

In [42]:
make_sink_prep(NUMS, CATS)

ColumnTransformer(transformers=[('spl',
                                 Pipeline(steps=[('spline',
                                                  SplineTransformer(include_bias=False,
                                                                    knots='quantile',
                                                                    n_knots=10)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'AvgChargePerMonth']),
                                ('lin', StandardScaler(),
                                 ['ChargeRes', 'EffectiveTenure',
                                  'ChargeResRate', 'ChargePerService']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               ha...
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'is_auto_payment', 'n_services']),
                                ('poly',
                                 Pipeline(steps=[('poly',
                                                  PolynomialFeatures(include_bias=False)),
                                                 ('sc', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges'])])

In [43]:
high_dim_preps = [
    ("interact",     make_spline_interact_prep),
    ("sink", make_sink_prep),
]
if HAS_CE:
    high_dim_preps.append(("multi_enc", make_multi_enc_spline_prep))

# L1 sweep
for prep_name, prep_fn in high_dim_preps:
    for C_val in [0.1, 0.5, 1.0]:
        lr_l1 = make_lr_l1(C=C_val)
        _, _, m_l1 = train_linear(
            X, y, test_use, prep_fn, lr_l1,
            exp_name=f"lr_{prep_name}_l1_C{C_val}")
        LOG.append(m_l1)

 lr_interact_l1_C0.1
    Features in  -> 26   |   Features out -> 1318
    Fold 1: AUC=0.876471


    Fold 2: AUC=0.785714
    Fold 3: AUC=0.876351
  CV AUC: 0.844984  |  mean±se: 0.846178 ± 0.024684  |  0.5s

 lr_interact_l1_C0.5


    Features in  -> 26   |   Features out -> 1318
    Fold 1: AUC=0.867059


    Fold 2: AUC=0.714286
    Fold 3: AUC=0.776711
  CV AUC: 0.782484  |  mean±se: 0.786018 ± 0.036209  |  0.6s

 lr_interact_l1_C1.0


    Features in  -> 26   |   Features out -> 1318
    Fold 1: AUC=0.847059


...

    Fold 2: AUC=0.765306


    Fold 3: AUC=0.764706
  CV AUC: 0.783134  |  mean±se: 0.785690 ± 0.016889  |  1.5s

 lr_multi_enc_l1_C1.0


    Features in  -> 26   |   Features out -> 129
    Fold 1: AUC=0.791765


    Fold 2: AUC=0.712018


    Fold 3: AUC=0.741897
  CV AUC: 0.751299  |  mean±se: 0.748560 ± 0.018992  |  1.7s



In [ ]:
# ElasticNet sweep (vary l1_ratio at best C candidates)
for prep_name, prep_fn in high_dim_preps:
    for l1r in [0.3, 0.5, 0.7]:
        lr_en = make_lr_enet(C=0.1, l1_ratio=l1r)
        _, _, m_en = train_linear(
            X, y, test_use, prep_fn, lr_en,
            exp_name=f"lr_{prep_name}_enet_l1r{l1r}")
        LOG.append(m_en)

## Category encoders experiments

Each encoder produces a different OOF vector from the same input data. We run with default C=1.0. If any enconder stands out, it can be further tuned with L1/ElasticNet sweeps.

In [ ]:
if HAS_CE:
    # (they all share the same structure)
    display(make_woe_spline_prep(NUMS, CATS))

    ce_preps = [
        ("woe",     make_woe_spline_prep),
        ("js",      make_js_spline_prep),
        ("cb",      make_cb_spline_prep),
        ("loo",     make_loo_spline_prep),
        ("binary",  make_binary_spline_prep),
        ("helmert", make_helmert_spline_prep),
        ("multi",   make_multi_enc_spline_prep),
    ]
    for name, prep_fn in ce_preps:
        _, _, m = train_linear(X, y, test_use, prep_fn, lr, exp_name=f"lr_{name}_spline")
        LOG.append(m)
else:
    print("  category_encoders not installed")


## Feature Engine experiments



In [ ]:
if HAS_FE:
    fe_preps = [
        ("freq",      make_freq_spline_prep),
        ("binned_te", make_binned_te_prep),
        ("binned_freq_te", make_binned_freq_te_prep),
        ("tree_disc", make_tree_disc_prep)]
    # Show first diagram
    display(make_freq_spline_prep(NUMS, CATS))

    for name, prep_fn in fe_preps:
        _, _, m = train_linear(X, y, test_use, prep_fn, lr, exp_name=f"lr_{name}")
        LOG.append(m)
else:
    print("  feature_engine not installed")

## Summary of experiments

The table below ranks all experiments by CV AUC:
- `cv_auc is the overall OOC AUC across all folds (the primary metric)
- `se` is the standard errorof fold-level AUCs. Two experiments differ meaningfully only if their AUC gap exceeds $\sqrt{2} \cdot SE$
- `n_features_out` number the columns after preprocessing. Higher features counts need stronger regularization.
- `elapsed` is the wall-clock time.

In [ ]:
summary = pd.DataFrame(LOG)[
    ["experiment_name", "cv_auc", "se", "n_features_out", "elapsed", "test_mode"]
].sort_values("cv_auc", ascending=False)
display(summary)

## Model selection for Ensemble

The purpose of this notebook is not to find one "winning" LR model, but to generate a diverse pool of OOF predictions that complement other models in the final ensemble.

We consider two models with AUC 0.91 and correlation of 0.92 more valuable to an ensemble than two models with AUC 0.915 and correlation of 0.99. Addind the later combination is redundant. They make very similar mistakes.

In [ ]:
import itertools
oof_files = sorted(SAVE_DIR.glob("*_oof.npy"))
oof_dict = {f.stem.replace("_oof", ""): np.load(f) for f in oof_files}
print(f"Total OOFs available: {len(oof_dict)}")

The selection strategy is:
1. Rank all OOFs
2. Compute correlations
3. Greedy selection: start with the best AUC, then add the next best that has correlation < threshold (0.98)
4. Save the selected OOFs

In [ ]:
oof_auc = {name: roc_auc_score(y, oof) for name, oof in oof_dict.items()}
ranked = sorted(oof_auc.items(), key=lambda x: -x[1])
print("\nTop 15 by CV AUC:")
for name, auc in ranked[:15]:
    print(f"  {auc:.6f}  {name}")

In [ ]:
top_names = [name for name, _ in ranked[:15]]
top_oofs = np.stack([oof_dict[n] for n in top_names])
corr = np.corrcoef(top_oofs)
mask = np.triu(np.ones_like(corr, dtype=bool))

import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, linewidth=0.5,
            xticklabels=top_names, yticklabels=top_names, ax=ax)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.title("OOF Correlation — Top 15 LR Models")
plt.tight_layout()
plt.show()

In [ ]:
CORR_THRESHOLD = 0.98

selected = [ranked[0][0]]  # best model (AUC)
for name, auc in ranked[1:]:
    oof_candidate = oof_dict[name]
    max_corr = max(
        np.corrcoef(oof_candidate, oof_dict[s])[0, 1]
        for s in selected
    )
    if max_corr < CORR_THRESHOLD:
        selected.append(name)

print(f"\nSelected {len(selected)} diverse OOFs (corr < {CORR_THRESHOLD}):")
for name in selected:
    print(f"  {oof_auc[name]:.6f}  {name}")

In [ ]:
print(f"\nFiles ready for ensemble pipeline:")
for name in selected:
    oof_path = SAVE_DIR / f"Selected_{name}_oof.npy"
    test_path = SAVE_DIR / f"Selected_{name}_test.npy"
    print(f"  {oof_path}  |  {test_path}")

## Submission

We create a LR submission based on simple probability average of the selected models. This is not expected to be competitive, its purpose is to validate CV-LB relation and sanity check test predictions.

In [ ]:
test_arrays = [np.load(SAVE_DIR / f"{name}_test.npy") for name in selected]

# Simple average
blend = np.mean(test_arrays, axis=0)

In [ ]:
sub["Churn"] = blend
sub.to_csv("submission.csv", index=False)

The training set has a churn rate of roughly 22.5% (the class prior). A well-calibrated model's average predicted probability should approximate the base rate. Also, a healthy range would be something where MAX $\ne$ MIN and where MIN $\ne$ 0 and MAX $\ne$ 1.

In [ ]:
print(f"Prediction range: [{blend.min():.4f}, {blend.max():.4f}]")
print(f"Prediction mean:  {blend.mean():.4f}")  